# Применение библиотеки Catboost к задаче предсказания рейтингов фильмов по ЭЭГ

Создадим матрицу признаков для модели. Для этого разделим по функциям на датафреймы и соединим их в один.

In [13]:
import mne
import yasa
import os
from tqdm import tqdm_notebook
import neurokit2 as nk
from scipy.signal import welch
import numpy as np
import timeit
import mne_features as mt
from tqdm.contrib.telegram import tqdm, trange

In [14]:
import warnings
import logging
import pandas as pd
warnings.filterwarnings('ignore')

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
logging.getLogger("mne").setLevel(logging.WARNING)

In [15]:
raw = mne.io.read_raw('Cuts_filtered_40_128/' + str(1) + '/001' + '_filtered_watch_' + str(1) + '_.fif')

# Neurokit2

In [16]:
sf = 128
win = int(4 * sf) 
chans = ['Fz', 'F3', 'F7','C3', 'T7', 'Pz', 'P3','P7', 'O1', 'Oz', 'O2','P4', 'P8', 'Cz','C4', 'T8', 'F4', 'F8']

In [ ]:
def first_der(x):
    return np.mean(np.abs(x[1:]-x[0:-1]))
 
for i in tqdm_notebook(range(7)):
    Subject = ''
    if len(str(i + 1)) == 1:
        Subject = '00' + str(i + 1)
    elif len(str(i + 1)) == 2:
        Subject = '0' + str(i + 1)
    for j in tqdm_notebook(range(7)):
        film = str(j + 1)
        raw = mne.io.read_raw('Cuts_filtered_40_128/' + str(int(Subject)) + '/00' + str(int(Subject)) + ('a' if i + 1 == 2 else '')  + '_filtered_watch_' + film + '_.fif')
        for k in tqdm_notebook(range(18)):
            if os.path.isfile('features_40_128_neurokit/S_' + str(int(Subject)) + '_F_' + film + '_' + chans[k] + '_.xlsx'):
                continue
            df_feat = {
                  # Entropy
                  #'complexity_apen': nk.complexity_apen(raw['Fz'][0][0] * 1e6, delay=delay, dimension=dim, tolerance=tol)[0],
                  'complexity_capen': nk.complexity_capen(raw[chans[k]][0][0] * 1e3)[0],
                  ###'complexity_cd': apply(nk.complexity_cd, 0, raw['Fz'][0][0] * 1e6)[0],
                  ###'complexity_cmse': nk.complexity_cmse(raw['Fz'][0][0] * 1e6)[0],
                  ##'complexity_cren': nk.complexity_cren(raw['Fz'][0][0] * 1e6)[0],
                  'complexity_dfa': nk.complexity_dfa(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_diffen': nk.complexity_diffen(raw[chans[k]][0][0] * 1e3)[0],
                  ##'complexity_fuzzycmse': nk.complexity_fuzzycmse(raw['Fz'][0][0] * 1e6)[0],
                  ###'complexity_fuzzyen': nk.complexity_fuzzyen(raw['Fz'][0][0] * 1e6)[0],
                  ###'complexity_fuzzymse': nk.complexity_fuzzymse(raw['Fz'][0][0] * 1e6, delay=delay, dimension=dim, tolerance=tol)[0],
                  ###'complexity_fuzzyrcmse': nk.complexity_fuzzyrcmse(raw['Fz'][0][0] * 1e6)[0],
                  'complexity_hjorth': nk.complexity_hjorth(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_hurst': nk.complexity_hurst(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_lempelziv': nk.complexity_lempelziv(raw[chans[k]][0][0] * 1e3)[0],
                  ###'complexity_lyapunov': nk.complexity_lyapunov(raw['Fz'][0][0] * 1e6)[0],
                  'complexity_mfdfa': nk.complexity_mfdfa(raw[chans[k]][0][0] * 1e3, multifractal=False)[0],
                  'complexity_mplzc': nk.complexity_mplzc(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_mse': nk.complexity_mse(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_mspe': nk.complexity_mspe(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_pe': nk.complexity_pe(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_plzc': nk.complexity_plzc(raw[chans[k]][0][0] * 1e3)[0],
                  ##'complexity_rcmse': nk.complexity_rcmse(raw['Fz'][0][0] * 1e6)[0],
                  'complexity_sampen': nk.complexity_sampen(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_se': nk.complexity_se(raw[chans[k]][0][0] * 1e3)[0],
                  'complexity_wpe': nk.complexity_wpe(raw[chans[k]][0][0] * 1e3)[0],
                  #'entropy_approximate': apply(nk.entropy_approximate, 0, raw[chans[k]][0][0] * 1e6)[0],
                  ##'entropy_coalition': apply(nk.entropy_coalition, 0, raw['Fz'][0][0] * 1e6)[0],
                  'entropy_cumulative_residual': nk.entropy_cumulative_residual(raw[chans[k]][0][0] * 1e3)[0],
                  'entropy_differential': nk.entropy_differential(raw[chans[k]][0][0] * 1e3)[0],
                  ###'entropy_fuzzy': apply(nk.entropy_fuzzy, 0, raw['Fz'][0][0] * 1e6)[0],
                  ##'entropy_multiscal': apply(nk.entropy_multiscal, 0, raw['Fz'][0][0] * 1e6)[0],
                  'entropy_permutation': nk.entropy_permutation(raw[chans[k]][0][0] * 1e3)[0],
                  #'entropy_sample': apply(nk.entropy_sample, 0, raw[chans[k]][0][0] * 1e6)[0],
                  'entropy_shannon': nk.entropy_shannon(raw[chans[k]][0][0] * 1e3)[0],
                  'entropy_spectral': nk.entropy_spectral(raw[chans[k]][0][0] * 1e3)[0],
                  'entropy_svd': nk.entropy_svd(raw[chans[k]][0][0] * 1e3)[0],
                  'fisher_information': nk.fisher_information(raw[chans[k]][0][0] * 1e3)[0],
                  ##'fractal_correlation': apply(nk.fractal_correlation, 0, raw['Fz'][0][0] * 1e6)[0],
                  #'fractal_dfa': apply(nk.fractal_dfa, 0, raw[chans[k]][0][0] * 1e6)[0],
                  ###'fractal_higuchi': apply(nk.fractal_higuchi, 0, raw['Fz'][0][0] * 1e6)[0],
                  'fractal_katz': nk.fractal_katz(raw[chans[k]][0][0] * 1e3)[0],
                  #'fractal_mandelbrot': apply(nk.fractal_mandelbrot, 0, raw['Fz'][0][0] * 1e6)[0],
                  #'fractal_mfdfa': nk.fractal_mfdfa(raw[chans[k]][0][0] * 1e6, multifractal=False)[0],
                  'fractal_petrosian': nk.fractal_petrosian(raw[chans[k]][0][0] * 1e3)[0],
                  'fractal_psdslope': nk.fractal_psdslope(raw[chans[k]][0][0] * 1e3)[0],
                  'fractal_sevcik': nk.fractal_sevcik(raw[chans[k]][0][0] * 1e3)[0],
                  ##'mutual_information': apply(nk.mutual_information, 0, raw['Fz'][0][0] * 1e6),
                  'first_der': first_der(raw[chans[k]][0][0] * 1e3)
                }
            df_feat = pd.DataFrame(df_feat, index=[0])
            df_feat.to_excel('features_40_128_neurokit/S_' + str(int(Subject)) + '_F_' + film + '_' + chans[k] + '_.xlsx', index=False)

# Yasa Bandpower

In [ ]:
for i in tqdm_notebook(range(7)):
    frames = []
    frames_rel = []
    for k in tqdm_notebook(range(8)):
        film = str(k + 1)
        Subject = ''
        if len(str(i + 1)) == 1:
            Subject = '00' + str(i + 1)
        elif len(str(i + 1)) == 2:
            Subject = '0' + str(i + 1)
        if i + 1 == 2:
            Subject += 'a'
        relax = mne.io.read_raw('Cuts_filtered_40_128/' + str(int(i + 1))  +  '/' + Subject + '_filtered_relax_before_' + film + '_.fif')
        watch = mne.io.read_raw('Cuts_filtered_40_128/' + str(int(i + 1))  + '/' + Subject + '_filtered_watch_' + film + '_.fif')
        relax.drop_channels(['Zygom', 'Corr', 'Mark', 'EDA', 'Pulse'])
        watch.drop_channels(['Zygom', 'Corr', 'Mark', 'EDA', 'Pulse'])
        if 'VEOG' in relax.ch_names:
            watch.drop_channels(['VEOG'])
            relax.drop_channels(['VEOG'])

        freqs_relax, psd_relax = welch(relax.get_data()  * 1e6, sf, nperseg=win, average='median')
        freqs_watch, psd_watch = welch(watch.get_data()  * 1e6, sf, nperseg=win, average='median')
        
        bp_relax = yasa.bandpower_from_psd(psd_relax, freqs_relax,chans, bands=
                                   [(4, 8, 'Theta'), (8, 12, 'Alpha'), (12, 30, 'Beta')], relative=False).drop(columns=['FreqRes', 'Relative'])
        bp_watch = yasa.bandpower_from_psd(psd_watch, freqs_watch,chans, bands=
                                   [(4, 8, 'Theta'), (8, 12, 'Alpha'), (12, 30, 'Beta')], relative=False).drop(columns=['FreqRes', 'Relative'])
    
        p_relax = yasa.bandpower(relax,relax.info['sfreq'], bands=
                                   [(4, 8, 'Theta_ratio'), (8, 12, 'Alpha_ratio'), (12, 30, 'Beta_ratio')], relative=True).drop(columns=['FreqRes', 'Relative',  'TotalAbsPow'])
        p_watch = yasa.bandpower(watch, watch.info['sfreq'], bands=
                                   [(4, 8, 'Theta_ratio'), (8, 12, 'Alpha_ratio'), (12, 30, 'Beta_ratio')], relative=True).drop(columns=['FreqRes', 'Relative',  'TotalAbsPow'])
        for j in tqdm_notebook(range(18)):
            df_relax = pd.DataFrame(bp_relax.loc[j]).transpose()
            df_watch = pd.DataFrame(bp_watch.loc[j]).transpose()
            ## Удаление дефективных каналов
            if df_relax['TotalAbsPow'].values > 1000 or df_watch['TotalAbsPow'].values > 1000:
                continue
            dr = pd.DataFrame(p_relax.loc[relax.ch_names[j]]).transpose()
            dr.rename(index={relax.ch_names[j]:j}, inplace=True)
            df_relax = df_relax.join(dr)
            dw = pd.DataFrame(p_watch.loc[watch.ch_names[j]]).transpose()
            dw.rename(index={relax.ch_names[j]:j}, inplace=True)
            df_watch = df_watch.join(dw)
            df_relax.rename(index={j:'F' + film + '_relax'}, inplace=True)
            df_watch.rename(index={j:'F' + film + '_watch'}, inplace=True)

            df_rel = {
                'Theta_watch-relax' : float(df_watch['Theta']) - float(df_relax['Theta']),
                'Alpha_watch-relax' : float(df_watch['Alpha']) - float(df_relax['Alpha']),
                'Beta_watch-relax' : float(df_watch['Beta']) - float(df_relax['Beta']),
            }
            df_rel = pd.DataFrame(df_rel, index=['F' + film + '_' + watch.ch_names[j]])
            df_div = {
                'Beta/Alpha' : float(df_rel['Beta_watch-relax']) / float(df_rel['Alpha_watch-relax']),
                'Beta/(Alpha + Theta)' : float(df_rel['Beta_watch-relax']) / (float(df_rel['Alpha_watch-relax'] + float(df_rel['Theta_watch-relax']))),
            }
            df_div = pd.DataFrame(df_div, index=['F' + film + '_' + watch.ch_names[j]])
            df_all = df_rel.join(df_div)
            frames_rel.append(df_all)
            frames.append(df_relax)
            frames.append(df_watch)
    result_rel = pd.concat(frames_rel)
    if not os.path.exists('BandPower/Excel/'):
        os.makedirs('BandPower/Excel/')
    result_rel.to_excel('BandPower/Excel/S' + str(i + 1) + '_ratio.xlsx')
    result = pd.concat(frames)
    if not os.path.exists('BandPower/Excel/'):
        os.makedirs('BandPower/Excel/')
    result.to_excel('BandPower/Excel/S' + str(i + 1) + '.xlsx')

In [ ]:
ind = 0
for i in tqdm_notebook(range(7)):
    Subject = ''
    if len(str(i + 1)) == 1:
        Subject = '00' + str(i + 1)
    elif len(str(i + 1)) == 2:
        Subject = '0' + str(i + 1)
    if i + 1 == 2:
        Subject += 'a'
    df_rel = pd.read_excel('BandPower/Excel/S' + str(i + 1) + '_ratio.xlsx')
    df = pd.read_excel('BandPower/Excel/S' + str(i + 1) + '.xlsx')
    first = 0
    second = 1
    for j in tqdm_notebook(range(8)):
        film = str(j + 1)
        raw = mne.io.read_raw('Cuts_filtered_40_128/' + str(int(i + 1))  +  '/' + Subject + '_filtered_relax_before_' + film + '_.fif')
        for k in tqdm_notebook(range(18)):
            dp = df_rel.loc[(df_rel['Unnamed: 0'] == 'F' + str(j + 1) + '_' + chans[k])].rename(index={first:ind})
            d = df.loc[(df['Chan'] == chans[k]) & (df['Unnamed: 0'] == 'F' + str(j + 1) + '_watch')].rename(index={second:ind})
            if dp.empty and d.empty:
                continue
            d['Theta_watch-relax'] = float(dp['Theta_watch-relax'])
            d['Alpha_watch-relax'] = float(dp['Alpha_watch-relax'])
            d['Beta_watch-relax'] = float(dp['Beta_watch-relax'])
            d['Beta/Alpha'] = float(dp['Beta/Alpha'])
            d['Beta/(Alpha + Theta)'] = float(dp['Beta/(Alpha + Theta)'])
            #display(d)
            d.to_excel('Essentials/S' + str(i + 1) + 'F' + film + '_' + chans[k] + '.xlsx')
            first += 1
            second += 2
            ind += 1

In [ ]:
df = []
chans = ['Fz', 'F3', 'F7','C3', 'T7', 'Pz', 'P3','P7', 'O1', 'Oz', 'O2','P4', 'P8', 'Cz','C4', 'T8', 'F4', 'F8']
for i in tqdm_notebook(range(7)):
    for j in tqdm_notebook(range(8)):
        for k in tqdm_notebook(range(18)):
            if os.path.isfile('Essentials/S'  + str(i + 1) + 'F' + str(j + 1) + '_' + chans[k] + '.xlsx'):
                df.append(pd.read_excel('Essentials/S'  + str(i + 1) + 'F' + str(j + 1) + '_' + chans[k] + '.xlsx'))

In [ ]:
#df = pd.concat(df)
#df.drop(['Unnamed: 0'], axis=1).to_excel('Features_power_neurokit.xlsx', index=False)
df = pd.read_excel('Features_power_neurokit.xlsx')

# MNE-Features

In [ ]:
features_all = []
features_all_bi = []
for i in tqdm_notebook(range(7)):
    Subject = ''
    if len(str(i + 1)) == 1:
        Subject = '00' + str(i + 1)
        if i + 1 == 2:
            Subject += 'a'
    elif len(str(i + 1)) == 2:
        Subject = '0' + str(i + 1)
    for j in tqdm_notebook(range(8)):
        film = str(j + 1)
        if os.path.isfile('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'univ.xlsx') and os.path.isfile('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'biv.xlsx'):
            continue
        raw = mne.io.read_raw('Cuts_filtered_40_128/' + str(i + 1) + '/' + Subject + '_filtered_watch_' + film + '_.fif')
        if i + 1 > 5:
            raw.drop_channels([ 'Zygom','Corr','Mark','EDA','Pulse' , 'VEOG'])
        else:
            raw.drop_channels([ 'Zygom','Corr','Mark','EDA','Pulse'])


        data = raw.get_data() * 1e3
        data = data.reshape(1, data.shape[0] , data.shape[1])
        if not os.path.isfile('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'biv.xlsx'):
            funcs_bi = mne_features.bivariate.get_bivariate_funcs(raw.info['sfreq'])
            funcs_bi.pop('nonlin_interdep')
            X_bi = mne_features.feature_extraction.extract_features(data, raw.info['sfreq'], funcs_bi, ch_names=raw.ch_names, n_jobs=-1, return_as_df=True)
            X_bi.to_excel('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'biv.xlsx')
            features_all_bi.append(X_bi)
        if not os.path.isfile('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'univ.xlsx'):
            funcs = mne_features.univariate.get_univariate_funcs(raw.info['sfreq'])
            funcs.pop('svd_entropy')
            funcs.pop('app_entropy')
            funcs.pop('hjorth_complexity')
            funcs.pop('svd_fisher_info')
            funcs.pop('hurst_exp')
            funcs.pop('pow_freq_bands')
            funcs.pop('katz_fd')
            funcs.pop('spect_entropy')
            funcs.pop('svd_entropy')
            X = mne_features.feature_extraction.extract_features(data, raw.info['sfreq'], funcs, n_jobs=-1,ch_names=raw.ch_names, return_as_df=True)
            X.to_excel('Features_mnef_40_128/S' + str(i + 1) + 'F' + film + 'univ.xlsx')
            features_all.append(X)


In [ ]:
from collections import OrderedDict

In [ ]:
result = pd.DataFrame()
ind_ex = 0
temp = pd.DataFrame()
for i in tqdm_notebook(range(7)):
    for j in tqdm_notebook(range(8)):
        conc = pd.DataFrame()
        biv = pd.read_excel('Features_mnef_40_128/' + 'S' + str(i + 1) + 'F' + str(j + 1) + 'biv.xlsx')
        univ = pd.read_excel('Features_mnef_40_128/' + 'S' + str(i + 1) + 'F' + str(j + 1) + 'univ.xlsx')
        columns = [elem for elem in univ.columns if elem.split(':')[0] != 'Unnamed']
        columns_biv = [elem for elem in biv.columns if elem.split(':')[0] != 'Unnamed']
        temp_univ = pd.DataFrame()
        temp_biv = pd.DataFrame()
        reject = []
        for m in tqdm_notebook(range(18)):
            if not os.path.isfile('Essentials/S'  + str(i + 1) + 'F' + str(j + 1) + '_' + chans[m] + '.xlsx'):
                reject.append(m)
        for k in range(0, len(columns)):
            current_univ = columns[k]
            if k == len(columns) - 1:
                values_univ = univ.loc[2:, current_univ:].values[0]
                columns_univ = np.array(univ.loc[0:,  current_univ].dropna())[0]
            else:
                nxt_unive = columns[k + 1]
                values_univ = univ.loc[2:, current_univ : nxt_unive].drop([nxt_unive], axis=1).values
                columns_univ = np.array(univ.loc[0:,  current_univ : nxt_unive].drop([nxt_unive], axis=1).dropna())[0]
            if current_univ == 'pow_freq_bands':
                continue
            new_columns_univ = []
            if len(columns_univ) > 18:
                for l in range(len(columns_univ)):
                    if current_univ == 'teager_kaiser_energy':
                        new_columns_univ.append(current_univ + '_' + ''.join(columns_univ[l].split('_')[1:]))
                    else:
                        new_columns_univ.append(current_univ + '_' + columns_univ[l].split('_')[-1])
            df_univ = pd.DataFrame(values_univ.reshape((18, -1)), columns=list(dict.fromkeys(new_columns_univ)) if len(columns_univ) > 18 else [current_univ], index=[i for i in range(ind_ex, ind_ex + 18)])
            temp_univ = pd.concat([temp_univ, df_univ], axis=1)
            
        for p in range(0, len(columns_biv)):
            current_biv = columns_biv[p]
            if p == len(columns_biv) - 1:
                values_biv = biv.loc[2:, current_biv:].values[0]
                columns = np.array(biv.loc[0:,  current_biv].dropna())[0]
            else:
                nxt_biv = columns_biv[p + 1]
                values_biv = biv.loc[2:, current_biv : ].drop([nxt_biv], axis=1).values[0]
                columns = np.array(biv.loc[0:,  current_biv : ].drop([nxt_biv], axis=1).dropna())[0]
            new_columns_biv = []
            for l in range(len(chans)):
                new_columns_biv.append(current_biv + '_' + chans[l])
            matrix = np.zeros((18, 18))
            left = 0
            ind = 17
            for d in range(0, 18):
                for k in range(d, 18):
                    if  d != k:
                        matrix[d][k] = values_biv[left:ind][k - d - 1]
                        matrix[k][d] = values_biv[left:ind][k - d - 1]
                left = ind
                ind += 17 - d - 1

            df_biv = pd.DataFrame(matrix, columns=new_columns_biv, index=[i for i in range(ind_ex, ind_ex + 18)])
            if current_biv == 'time_corr':
                df_eig = pd.DataFrame(values_biv[ind + 1:len(values_biv)], columns=['eig'], index=[i for i in range(ind_ex, ind_ex + 18)])
                df_biv = pd.concat([df_biv, df_eig], axis=1)
            temp_biv = pd.concat([temp_biv, df_biv], axis=1)
        if len(reject) > 0:
            conc = pd.concat([temp_univ, temp_biv], axis=1).drop(np.array(reject) + ind_ex, axis=0)
        else:
            conc = pd.concat([temp_univ, temp_biv], axis=1)
        result = pd.concat([result, conc])
        ind_ex += 18
print(result.shape)

In [ ]:
#result.to_excel('mne_features_new.xlsx', index=False)

In [ ]:
#result.to_excel('Features_mne.xlsx', index=False)
result = pd.read_excel('mne_features_new.xlsx')
result

In [ ]:
data_all = pd.concat([df, result], axis=1)
data_all

In [ ]:
ind = []
i = 0
for i in tqdm_notebook(range(7)):
    for j in tqdm_notebook(range(8)):
        for k in tqdm_notebook(range(18)):
            if os.path.isfile('Essentials/S'  + str(i + 1) + 'F' + str(j + 1) + '_' + chans[k] + '.xlsx'):
                ind.append('S_'  + str(i + 1) + '_F_' + str(j + 1) + '_' + chans[k])
data_all['Unnamed: 0.1'] = ind
chans_d = {}
for i in range(len(chans)):
    chans_d[chans[i]] = i
data_all = data_all.drop(['Chan'], axis=1)
data_all

In [ ]:
labels = [7.4, 5.8, 4.6, 6, 6.6, 6, 5.4, 5.6]
data_all['film'] = [int(np.array(data_all['Unnamed: 0.1'])[i].split('_')[3]) for i in range(len(np.array(data_all['Unnamed: 0.1'])))]
data_all['Subj'] = [int(np.array(data_all['Unnamed: 0.1'])[i].split('_')[1].split('_')[0]) for i in range(len(np.array(data_all['Unnamed: 0.1'])))]
data_all['ch'] = [np.array(data_all['Unnamed: 0.1'])[i].split('_')[4] for i in range(len(np.array(data_all['Unnamed: 0.1'])))]
data_all['ch'] = [chans_d[np.array(data_all['ch'])[i]] for i in range(len(np.array(data_all['Unnamed: 0.1'])))]
data_all['labels'] = [labels[int(np.array(data_all['Unnamed: 0.1'])[i].split('_')[3]) - 1] for i in range(len(np.array(data_all['Unnamed: 0.1'])))]

In [ ]:
data_all

In [ ]:
data_all = data_all.drop(['Unnamed: 0.1'], axis=1)
data_all.to_excel('features_new_mne_features_2.xlsx', index=False)
data_all

# Обучение 

Начнем с поиска гиперпараметров

In [ ]:
%matplotlib qt

In [ ]:
data_all = pd.read_excel('features_new_mne_features_2.xlsx')
data_all

In [ ]:
from catboost import Pool, CatBoostRegressor, cv
import matplotlib

In [ ]:
import ipywidgets

In [ ]:
model = CatBoostRegressor(cat_features=['ch', 'film', 'Subj'],task_type='CPU')

labels = data_all['labels']
cv_data = data_all.drop(['labels'],axis=1)


cv_dataset = Pool(data=cv_data,
                  label=labels,
                  cat_features=['ch', 'film', 'Subj'])
grid = {
          'depth':[1, 3, 5, 7, 10],
          'iterations':[500, 1000],
          'learning_rate':[0.03, 0.01, 0.3], 
          'l2_leaf_reg':[1, 3, 5],
}

grid_search_result_noise = model.grid_search(grid, cv_dataset,
                                       verbose=False,
                                       shuffle=True,
                                       plot=True)

In [ ]:
grid_search_result_noice = {}
grid_search_result_noise['params'] = {
          'depth': 3,
          'iterations': 10000,
          'learning_rate': 0.03, 
          'l2_leaf_reg':1,    
}

In [ ]:
grid_search_result_noise['params']['loss_function'] = 'RMSE'

In [ ]:
grid_search_result_noise['params']

# Кросс - валидация

In [ ]:
labels = data_all['labels']
cv_data = data_all.drop(['labels'],axis=1)


cv_dataset = Pool(data=cv_data,
                  label=labels,
                   cat_features=['ch', 'film', 'Subj'])


scores = cv(cv_dataset,
            grid_search_result_noise['params'],
            verbose=False,
            fold_count=5, 
            plot="True",
            as_pandas=True)

In [ ]:
scores.to_excel('Catboost/cv.xlsx')
scores

Выберем 1 фильм для предсказания

In [ ]:
ind = data_all['film'] == 2
X_test = data_all[ind]
X_train = data_all.loc[(data_all['film'] == 5) | (data_all['film'] == 1)  | (data_all['film'] == 3) | (data_all['film'] == 4) | (data_all['film'] == 6) | (data_all['film'] == 7) | (data_all['film'] == 8)] 

In [ ]:
X_train = X_train.sample(frac=1)
X_test = X_test.sample(frac=1)
y_train = X_train['labels']
y_test = X_test['labels']
test_film = X_test['film']
test_subj = X_test['Subj']
X_train = X_train.drop(['labels'], axis=1)
X_test = X_test.drop(['labels'], axis=1)

In [ ]:
cat_features = ['ch', 'film', 'Subj']

In [ ]:
train_pool = Pool(data=X_train, label=y_train,  cat_features=cat_features)

In [ ]:
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
model_fast = CatBoostRegressor(
    **grid_search_result_noise['params'],
    task_type='CPU',
    od_type='Iter',
    od_wait = 1000,
    bootstrap_type='Bernoulli',
    one_hot_max_size=6,
    rsm=0.3,
    max_ctr_complexity=1,
)

In [ ]:
model_fast.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    use_best_model=True,
    eval_set=test_pool,
    plot=True,
    logging_level='Silent'
)

In [ ]:
model_tunned = CatBoostRegressor(
    **grid_search_result_noise['params'],
    cat_features=cat_features,
    bagging_temperature=1,
    random_strength=1,
    one_hot_max_size=2,
    task_type='CPU',
    od_type='Iter',
    od_wait = 2000,
)

In [ ]:
model_tunned.fit(train_pool, use_best_model=True, eval_set=test_pool, plot=True, logging_level='Silent')

In [ ]:
model_tunned.get_all_params()

In [ ]:
model_tunned.tree_count_ 

In [ ]:
model_best = CatBoostRegressor(
    iterations=int(model_tunned.tree_count_ * 1.2),
    cat_features=cat_features,
    bagging_temperature=1,
    random_strength=1,
    one_hot_max_size=2,
    task_type='CPU',
    od_type='Iter',
    od_wait = 3000,
    leaf_estimation_iterations=10,
)

In [ ]:
model_best.fit(train_pool, plot=True, logging_level='Silent')

In [ ]:
preds = model_best.predict(X_test)

In [ ]:
preds

Усредним полученные предсказания по всем каналам для каждого испытуемого

In [ ]:
pred = [0. for i in range(7)]
num_chans = [0. for i in range(7)]
for i in range(len(preds)):
    pred[np.array(test_subj)[i] - 1] += (preds)[i]
    num_chans[np.array(test_subj)[i] - 1] += 1
res = []
print('Chans', num_chans)
for i in range(len(pred)):
    res.append(pred[i] / num_chans[i])
rmse = model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]
rmse_normalized = rmse / (7.4 - 4.6)
print('Result', res)
print('Mean: {}, RMSE : {}, Normalized RMSE: {}'.format(np.mean(res), rmse, rmse_normalized))


# Анализ признаков

In [ ]:
from catboost import CatBoostRegressor, EShapCalcType, EFeaturesSelectionAlgorithm
import shap

In [ ]:
feature_importance = model_best.get_feature_importance(data=test_pool,
                       type='LossFunctionChange',
                       reference_data=None,
                       prettified=True,
                       thread_count=-1,
                       verbose=False)

In [ ]:
feature_importance.to_excel('Catboost/lossFC_2_imp.xlsx')

In [ ]:
feature_importance

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def get_feature_imp_plot(method, model):
    
    if method == "ShapeValues":
        shap_values = model.get_feature_importance(Pool(X_test, label=y_test, cat_features=['ch', 'film', 'Subj']), 
                                                                     type="ShapValues")
        shap_values = shap_values[:,:-1]
        shap.summary_plot(shap_values, X_test) 
        
    else:
        fi = model.get_feature_importance(Pool(X_test, label=y_test, cat_features=['ch', 'film', 'Subj']), 
                                                                     type=method)
        
    if method != "ShapeValues":
        feature_score = pd.DataFrame(list(zip(X_test.dtypes.index, fi )),
                                        columns=['Feature','Score'])

        feature_score = feature_score.sort_values(by='Score', ascending=False, inplace=False, kind='quicksort', na_position='last')

        plt.rcParams["figure.figsize"] = (12,7)
        ax = feature_score.plot('Feature', 'Score', kind='bar', color='c')
        ax.set_title("Feature Importance using {}".format(method), fontsize = 14)
        ax.set_xlabel("features")

In [ ]:
get_feature_imp_plot(method="LossFunctionChange", model=model_best)

In [ ]:
get_feature_imp_plot(method="PredictionValuesChange", model=model_best)

In [ ]:
get_feature_imp_plot(method="ShapeValues",model=model_best)

In [ ]:
def select_features_adult(X_train, y_train, num_features, algorithm: EFeaturesSelectionAlgorithm, steps: int = 1,):
    print('Algorithm:', algorithm)
    model_1 = CatBoostRegressor(
        **grid_search_result_noise['params'],
        task_type='CPU',
        od_type='Iter',
        od_wait = 1000,
        bootstrap_type='Bernoulli',
        one_hot_max_size=6,
        rsm=0.3,
        max_ctr_complexity=1,
    )
    summary = model_1.select_features(
        train_pool,
        eval_set=test_pool,
        features_for_select=list(range(train_pool.num_col())),
        num_features_to_select=num_features,
        steps=steps,
        algorithm=algorithm,
        shap_calc_type=EShapCalcType.Regular,
        train_final_model=True,
        logging_level='Silent',
        plot=False
    )
    cfeatures = ['ch', 'film', 'Subj']
    cat_features = [cfeatures[i]  for i in range(len(cfeatures)) if cfeatures[i] in summary['selected_features_names']]
    if len(cat_features) == 0:
        cat_features = None
    model_2 = CatBoostRegressor(
        **grid_search_result_noise['params'],
        cat_features=cat_features,
        bagging_temperature=1,
        random_strength=1,
        one_hot_max_size=2,
        task_type='CPU',
        od_type='Iter',
        od_wait = 2000,
    )
    model_2.fit(X_train[summary['selected_features_names']], y_train, eval_set=Pool(X_test[summary['selected_features_names']], y_test, cat_features=cat_features), verbose = False, plot=False)
    rmse = model_2.eval_metrics(Pool(X_test[summary['selected_features_names']], y_test, cat_features=cat_features), ['RMSE'])['RMSE'][-1]
    rmse_normalized = rmse / (7.4 - 4.6)
    print('{} : RMSE : {}, Normalized RMSE: {}'.format(num_features, rmse, rmse_normalized))

    
    print('Number of features selected : ', len(summary['selected_features_names']) , 'Selected features:', summary['selected_features_names'])
    return summary

In [ ]:
for i in tqdm_notebook(range(1, 10)):
    summary = select_features_adult(X_train, y_train, i, algorithm=EFeaturesSelectionAlgorithm.RecursiveByLossFunctionChange, steps=3)

In [ ]:
for i in tqdm_notebook(range(1, 10)):
    summary = select_features_adult(X_train, y_train, i, algorithm=EFeaturesSelectionAlgorithm.RecursiveByPredictionValuesChange, steps=3)

In [ ]:
for i in tqdm_notebook(range(5, 31)):
    summary = select_features_adult(X_train, y_train, i, algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues, steps=3)

In [ ]:
selected_features_names = ['Theta', 'Beta', 'TotalAbsPow', 'teager_kaiser_energy_5std', 'wavelet_coef_energy_1', 'wavelet_coef_energy_2', 'max_cross_corr_C3', 'max_cross_corr_Oz', 'phase_lock_val_P7', 'phase_lock_val_O1', 'spect_corr_P4', 'time_corr_O1']

In [ ]:
pd.DataFrame(selected_features_names).to_excel('Catboost/selected_features.xlsx')

In [ ]:
data_all_2 = pd.read_excel('features_new_mne_features_2.xlsx')
data_all_2 = data_all_2[selected_features_names + ['labels']]
data_all_2

In [ ]:
X_tr = X_train[selected_features_names].reset_index(drop=True)
X_tr

In [ ]:
X_t = X_test[selected_features_names].reset_index(drop=True)

In [ ]:
X_t

In [ ]:
y_tr = y_train.reset_index(drop=True)

In [ ]:
labels = data_all_2['labels']
cv_data = data_all_2[selected_features_names]

cfeatures = ['ch', 'film', 'Subj']
cat_features = [cfeatures[i]  for i in range(len(cfeatures)) if cfeatures[i] in selected_features_names]
if len(cat_features) == 0:
    cat_features = None

model_features = CatBoostRegressor(task_type='CPU', cat_features=cat_features)
cv_dataset = Pool(data=cv_data,
                  label=labels,
                  cat_features=cat_features)
grid = {
          'depth':[1, 3, 5, 7, 10],
          'iterations':[500, 1000],
          'learning_rate':[0.03, 0.01, 0.3], 
          'l2_leaf_reg':[1, 3, 5],    
}

grid_search_result = model_features.grid_search(grid,cv_dataset,
                                       verbose=False,
                                       shuffle=True,
                                       plot=True)

In [ ]:
cat_features

In [ ]:
grid_search_result['params']['loss_function'] = 'RMSE'
grid_search_result['params']

In [ ]:
labels = data_all['labels']
cv_data = data_all[selected_features_names]


cv_dataset = Pool(data=cv_data,
                  label=labels,
                   cat_features=cat_features)


scores = cv(cv_dataset,
            grid_search_result['params'],
            shuffle=True,
            fold_count=5, 
            plot="True",
            as_pandas=True)

In [ ]:
scores.to_excel('Catboost/cv_2.xlsx')
scores

In [ ]:
cb = CatBoostRegressor(
    **grid_search_result['params'],
    cat_features=cat_features,
    bagging_temperature=1,
    random_strength=1,
    one_hot_max_size=2,
    task_type='CPU',
    od_type='Iter',
    od_wait = 2000,
)

In [ ]:
cb.fit(
    X_tr[selected_features_names],
    y_tr,
    use_best_model=True,
    eval_set=Pool(X_t, y_test, cat_features),
    plot=True
)

In [ ]:
indices, scores = cb.get_object_importance(
    Pool(X_t, y_test, cat_features=cat_features),
    Pool(X_tr, y_tr, cat_features=cat_features),
    importance_values_sign='Positive',
    update_method='AllPoints'
    # Positive values means that the optimized metric
                                      # value is increase because of given train objects.
                                      # So here we get the indices of bad train objects.
)

In [ ]:
def train_and_print_score(train_indices, remove_object_count):
    cb.fit(X_tr[train_indices], y_tr[train_indices],verbose=False, use_best_model=True, eval_set=Pool(X_t, y_test, cat_features))
    metric_value = cb.eval_metrics(Pool(X_t, y_test, cat_features), ['RMSE'])['RMSE'][-1]
    s = 'RMSE on validation datset when {} harmful objects from train are dropped: {}'
    print(s.format(remove_object_count, metric_value))

batch_size = 10
train_indices = np.full(X_train.shape[0], True)
train_and_print_score(train_indices, 0)
for batch_start_index in range(0, 150, batch_size):
    train_indices[indices[batch_start_index:batch_start_index + batch_size]] = False
    train_and_print_score(train_indices, batch_start_index + batch_size)

In [ ]:
drop = 150

In [ ]:
X_tr = X_tr.drop(indices[:drop], axis=0)
X_tr

In [ ]:
y_tr = y_tr.drop(indices[:drop], axis=0)
y_tr

In [ ]:
cv_dataset = Pool(data=X_tr,
                  label=y_tr,
                   cat_features=cat_features)


scores = cv(cv_dataset,
            grid_search_result['params'],
            fold_count=5,
            verbose=False,
            plot="True",
            as_pandas=True)

In [ ]:
scores.to_excel('Catboost/cv_3.xlsx')
scores

In [ ]:
model_filtered_fast = CatBoostRegressor(
    **grid_search_result['params'],
    task_type='CPU',
    od_type='Iter',
    od_wait = 1000,
    bootstrap_type='Bernoulli',
    one_hot_max_size=6,
    rsm=0.3,
    max_ctr_complexity=1,
)

In [ ]:
model_filtered_fast.fit(
    X_tr,
    y_tr,
    cat_features=cat_features,
    use_best_model=True,
    eval_set=Pool(X_t, y_test, cat_features=cat_features),
    plot=True,
    logging_level='Silent'
)

In [ ]:
model_filtered_tunned = CatBoostRegressor(
    **grid_search_result['params'],
    cat_features=cat_features,
    bagging_temperature=1,
    random_strength=1,
    one_hot_max_size=2,
    task_type='CPU',
    od_type='Iter',
    od_wait = 2000,
)

In [ ]:
model_filtered_tunned.fit(
    X_tr,
    y_tr,
    cat_features=cat_features,
    use_best_model=True,
    eval_set=Pool(X_t, y_test, cat_features=cat_features),
    plot=True,
    logging_level='Silent'
)

In [ ]:
model_filtered_best = CatBoostRegressor(
    iterations=int(model_filtered_tunned.tree_count_ * 1.2),
    cat_features=cat_features,
    bagging_temperature=1,
    random_strength=1,
    one_hot_max_size=2,
    task_type='CPU',
    od_type='Iter',
    od_wait = 3000,
    leaf_estimation_iterations=10,
)

In [ ]:
model_filtered_best.fit(
    X_tr,
    y_tr,
    plot=True,
    logging_level='Silent'
)

In [ ]:
preds = model_filtered_best.predict(X_t)

In [ ]:
preds

In [ ]:
pred = [0. for i in range(7)]
num_chans = [0. for i in range(7)]
for i in range(len(preds)):
    pred[np.array(test_subj)[i] - 1] += (preds)[i]
    num_chans[np.array(test_subj)[i] - 1] += 1
res = []
print('Chans', num_chans)
for i in range(len(pred)):
    res.append(pred[i] / num_chans[i])
rmse = model_filtered_best.eval_metrics(Pool(X_t, y_test, cat_features=cat_features), ['RMSE'])['RMSE'][-1]
rmse_normalized = rmse / (7.4 - 4.6)
print('Result', res)
print('Mean: {}, RMSE : {}, Normalized RMSE: {}'.format(np.mean(res), rmse, rmse_normalized))


In [ ]:
X_tr.to_excel('Catboost/X_train.xlsx')
X_t.to_excel('Catboost/X_test.xlsx')
y_tr.to_excel('Catboost/y_train.xlsx')
y_test.to_excel('Catboost/y_test.xlsx')